In [2]:
import torch
import testdata
from _em import _EM_step_full_stable, fit_EM_iter

In [3]:
%load_ext autoreload

In [4]:
%autoreload 2

Generate fake data

In [70]:
params = {'d': 10, 'k': [5, 3, 4], 'p': [15, 13, 14], 'n': 5000,'sigsq': [0.3, 0.7, 0.5]}

Y, W, L, Phi = testdata.simulate_data(params, private_var=True, verbose=True)
W_init, L_init, Phi_init = testdata.initialize_params(W, L, Phi, private_var=True)

Y = Y.T

Y = WZ + LX + E


In [71]:
Sigma_hat = Y.T @ Y / Y.shape[0]

Ground truth

In [72]:
for W_m in W:
    print((W_m @ W_m.T)[:5, :5])

tensor([[ 12.8786,  -1.3312,   0.4138, -10.5231,   2.4906],
        [ -1.3312,   2.0729,  -2.7896,   3.0509,  -1.1204],
        [  0.4138,  -2.7896,   7.3893,  -3.7282,   0.1549],
        [-10.5231,   3.0509,  -3.7282,  17.4196,  -4.1377],
        [  2.4906,  -1.1204,   0.1549,  -4.1377,   4.5822]])
tensor([[12.9950, -8.7043,  1.5269, -6.6354, -1.8701],
        [-8.7043, 11.7268,  0.5824,  4.1213, -2.6713],
        [ 1.5269,  0.5824,  5.6008, -2.0348, -3.3226],
        [-6.6354,  4.1213, -2.0348,  9.4010,  0.9237],
        [-1.8701, -2.6713, -3.3226,  0.9237,  7.5636]])
tensor([[15.8767, -3.0576, -8.1599,  5.6757,  0.2537],
        [-3.0576, 10.9647,  0.9861, -0.3930,  1.7829],
        [-8.1599,  0.9861,  8.4417, -1.8650,  1.8916],
        [ 5.6757, -0.3930, -1.8650,  9.0264,  0.4484],
        [ 0.2537,  1.7829,  1.8916,  0.4484, 11.5859]])


In [73]:
for L_m in L:
    print((L_m @ L_m.T)[:5, :5])

tensor([[ 7.3384,  2.5604, -3.5822,  3.8962, -3.7054],
        [ 2.5604,  2.6291, -3.0680,  3.0547, -0.5803],
        [-3.5822, -3.0680,  5.2785, -4.0331,  0.7821],
        [ 3.8962,  3.0547, -4.0331,  3.8069, -1.1870],
        [-3.7054, -0.5803,  0.7821, -1.1870,  2.3381]])
tensor([[ 6.1216,  3.7147, -1.3169, -0.5018,  1.9652],
        [ 3.7147,  4.7635, -0.2850, -2.7109,  1.3772],
        [-1.3169, -0.2850,  0.3939, -0.3643, -0.4353],
        [-0.5018, -2.7109, -0.3643,  2.4298, -0.5349],
        [ 1.9652,  1.3772, -0.4353, -0.5349,  1.1223]])
tensor([[18.8776, -5.7316, -2.0603, -2.5124, -7.2526],
        [-5.7316,  6.8570, -3.5628,  1.4425,  1.0538],
        [-2.0603, -3.5628, 10.5819, -0.1122,  4.2651],
        [-2.5124,  1.4425, -0.1122,  0.4465,  0.8485],
        [-7.2526,  1.0538,  4.2651,  0.8485,  4.0100]])


In [74]:
for Phi_m in Phi:
    print((Phi_m)[:5, :5])

tensor([[0.3000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3000]])
tensor([[0.7000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.7000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7000]])
tensor([[0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5000]])


Test EM step with complete data

In [75]:
W_new, L_new, Phi_new, _, _ = fit_EM_iter(Y, Sigma_hat, W_init, L_init, Phi_init, maxit = 1000)

In [76]:
for W_m in W_new:
    print((W_m @ W_m.T)[:5, :5])

tensor([[ 12.5225,  -1.2553,   0.3421, -10.4120,   2.5785],
        [ -1.2553,   2.2890,  -2.9304,   3.3441,  -1.2082],
        [  0.3421,  -2.9304,   7.3655,  -3.9761,   0.3443],
        [-10.4120,   3.3441,  -3.9761,  18.6219,  -4.6509],
        [  2.5785,  -1.2082,   0.3443,  -4.6509,   4.6335]])
tensor([[13.5777, -9.0234,  1.4846, -6.7020, -1.7598],
        [-9.0234, 11.6286,  0.5947,  4.0840, -2.6828],
        [ 1.4846,  0.5947,  5.6995, -1.8448, -3.2445],
        [-6.7020,  4.0840, -1.8448,  9.2725,  0.7522],
        [-1.7598, -2.6828, -3.2445,  0.7522,  7.5311]])
tensor([[16.1148, -3.4838, -8.7848,  5.4959, -0.5964],
        [-3.4838, 12.1699,  0.8916, -0.3645,  2.0325],
        [-8.7848,  0.8916,  9.0885, -1.9827,  1.9581],
        [ 5.4959, -0.3645, -1.9827,  8.8998,  0.2797],
        [-0.5964,  2.0325,  1.9581,  0.2797, 11.9979]])


In [77]:
for L_m in L_new:
    print((L_m @ L_m.T)[:5, :5])

tensor([[ 7.4503,  2.6349, -3.7778,  3.8850, -3.7116],
        [ 2.6349,  2.5925, -3.1084,  2.9821, -0.6085],
        [-3.7778, -3.1084,  5.4898, -4.0118,  0.8732],
        [ 3.8850,  2.9821, -4.0118,  3.6516, -1.1793],
        [-3.7116, -0.6085,  0.8732, -1.1793,  2.3118]])
tensor([[ 6.4644,  3.8373, -1.3334, -0.5044,  2.1040],
        [ 3.8373,  4.7149, -0.2599, -2.6261,  1.4512],
        [-1.3334, -0.2599,  0.3938, -0.3876, -0.4262],
        [-0.5044, -2.6261, -0.3876,  2.3499, -0.5609],
        [ 2.1040,  1.4512, -0.4262, -0.5609,  1.1672]])
tensor([[19.1254, -5.7113, -2.3553, -2.6053, -7.4728],
        [-5.7113,  6.7105, -3.4796,  1.4537,  1.0523],
        [-2.3553, -3.4796, 10.7399, -0.0700,  4.5125],
        [-2.6053,  1.4537, -0.0700,  0.4652,  0.9024],
        [-7.4728,  1.0523,  4.5125,  0.9024,  4.2062]])


In [78]:
for Phi_m in Phi_new:
    print((Phi_m)[:5, :5])

tensor([[0.3170, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.2958, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2782, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3086, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3150]])
tensor([[0.5192, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6805, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6906, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.6683, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.6875]])
tensor([[0.4860, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4492, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5334, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.5038, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5306]])


Test EM step with missing data (each sample missing max 1 mode)

In [116]:
Y_miss = Y.clone().detach()

In [117]:
Y_miss[1000:1025, :params['p'][0]] = float('nan')
Y_miss[1025:1075, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
Y_miss[1500:1550, params['p'][0]+params['p'][1]:] = float('nan')

Y_miss[1020:1030, params['p'][0]-3:params['p'][0]+3]

tensor([[    nan,     nan,     nan, -4.7369,  3.6317, -1.1304],
        [    nan,     nan,     nan, 15.6481,  1.2268,  5.5821],
        [    nan,     nan,     nan, -1.0027, -1.7510,  1.7807],
        [    nan,     nan,     nan, -2.8922,  3.7636,  5.6013],
        [    nan,     nan,     nan,  7.5011, -1.3625,  9.3244],
        [-1.1836, -0.6728, -2.1152,     nan,     nan,     nan],
        [-0.4631, -1.6067,  2.0418,     nan,     nan,     nan],
        [-8.3885, -2.8173, -3.4735,     nan,     nan,     nan],
        [-0.8251,  2.9691, -3.7897,     nan,     nan,     nan],
        [-1.3127,  2.5663,  4.8090,     nan,     nan,     nan]])

In [118]:
Y_miss[1495:1505, params['p'][1]-3:params['p'][1]+3]

tensor([[ 2.2145,  4.0129, -4.1522, -1.1282,  2.3907, -9.8586],
        [ 1.4810,  0.9793,  3.1720, -3.2174, -3.7450, -3.9431],
        [-0.7133,  1.2480, -0.5098,  3.7531,  3.1531, -6.6442],
        [-3.6249, -3.8704, -0.4659,  1.9463, -2.6342,  6.6251],
        [ 4.5398,  1.1661,  3.2263, -5.2044,  0.2658, -0.3478],
        [ 3.4783, -2.3834, -0.4023,     nan,     nan,     nan],
        [ 0.1731,  0.1142,  0.7257,     nan,     nan,     nan],
        [-0.3718,  0.7807, -2.7112,     nan,     nan,     nan],
        [ 1.9415,  2.1648,  3.5468,     nan,     nan,     nan],
        [-3.9015, -2.0470, -3.9217,     nan,     nan,     nan]])

In [119]:
W_new, L_new, Phi_new, _, _ = fit_EM_iter(Y_miss, Sigma_hat, W_init, L_init, Phi_init, maxit = 1000, impute_modes = True)

In [120]:
for W_m in W_new:
    print((W_m @ W_m.T)[:5, :5])

tensor([[ 7.6096,  3.5457,  0.4034, -0.4018,  1.6839],
        [ 3.5457, 11.5820, -3.3375,  5.9190,  1.5057],
        [ 0.4034, -3.3375,  8.7881, -2.7597,  3.1580],
        [-0.4018,  5.9190, -2.7597, 11.1078, -0.1702],
        [ 1.6839,  1.5057,  3.1580, -0.1702, 16.4197]])
tensor([[28.9639, -1.1958, -6.4173, -8.0007, -4.0370],
        [-1.1958, 11.9165,  6.5737,  3.2529, -3.7754],
        [-6.4173,  6.5737, 20.3224,  4.5034, -5.6889],
        [-8.0007,  3.2529,  4.5034,  7.5678, -2.0973],
        [-4.0370, -3.7754, -5.6889, -2.0973,  6.3797]])
tensor([[12.9598, -5.5152,  2.6866, -1.3225,  0.5286],
        [-5.5152, 12.1446, -1.1022,  3.4042,  1.6181],
        [ 2.6866, -1.1022,  9.7183, -1.7317,  5.1253],
        [-1.3225,  3.4042, -1.7317,  2.0794, -1.2020],
        [ 0.5286,  1.6181,  5.1253, -1.2020,  7.9692]])


In [121]:
for L_m in L_new:
    print((L_m @ L_m.T)[:5, :5])

tensor([[ 5.2897, -2.1766,  4.3091,  3.0840, -0.4579],
        [-2.1766,  3.9314, -1.1414, -0.3086, -1.3465],
        [ 4.3091, -1.1414,  5.5483,  3.6171, -0.6135],
        [ 3.0840, -0.3086,  3.6171,  7.8940, -3.0714],
        [-0.4579, -1.3465, -0.6135, -3.0714,  2.0210]])
tensor([[ 1.0986,  1.6520, -1.4669, -1.3253,  0.0098],
        [ 1.6520,  3.5493, -3.6506, -3.1780,  1.3402],
        [-1.4669, -3.6506,  3.9605,  3.5209, -1.9957],
        [-1.3253, -3.1780,  3.5209,  3.4076, -2.1162],
        [ 0.0098,  1.3402, -1.9957, -2.1162,  2.4581]])
tensor([[ 6.0991,  3.8208, -1.4286,  0.5871, -0.6367],
        [ 3.8208,  4.2346,  0.3049,  1.5314,  2.2305],
        [-1.4286,  0.3049,  4.0028,  0.0430,  1.6659],
        [ 0.5871,  1.5314,  0.0430,  1.3834,  2.0825],
        [-0.6367,  2.2305,  1.6659,  2.0825,  4.2469]])


In [91]:
for Phi_m in Phi_new:
    print(Phi_m[:8,:8])

tensor([[0.8772, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3417, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5643, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.6574, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3342, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4354, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5517, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5269]])
tensor([[1.3258, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 1.0940, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7505, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.8061, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.8476, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.00

In [97]:
# MSE for each feature in Phi
torch.cat([
    torch.diag(Phi_m) for Phi_m in Phi_new
]) - torch.cat([
    torch.diag(Phi_m) for Phi_m in Phi
])**2

tensor([0.7872, 0.2517, 0.4743, 0.5674, 0.2442, 0.3454, 0.4617, 0.4369, 0.2214,
        0.3340, 0.4919, 0.2727, 0.3953, 0.6600, 0.8613, 0.8358, 0.6040, 0.2605,
        0.3161, 0.3576, 0.5059, 0.3583, 0.5202, 0.4007, 0.5232, 0.2749, 0.2360,
        0.3921, 1.5911, 0.6030, 0.9252, 0.3796, 0.6616, 0.4789, 0.4933, 0.7385,
        1.0568, 0.5042, 0.7882, 0.3175, 0.5899, 0.6687])

In [96]:
# average MSE for Phi
torch.mean(torch.cat([
    torch.diag(Phi_m) for Phi_m in Phi_new
]) - torch.cat([
    torch.diag(Phi_m) for Phi_m in Phi
])**2).item()


0.5282648205757141

In [101]:
# correlation of Phi 
torch.corrcoef(
    torch.stack([
        torch.cat([torch.diag(Phi_m) for Phi_m in Phi]).flatten(),
        torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new]).flatten()
    ], dim=0)
)[0,1].item()

0.5277745127677917

Test EM step with missing data (samples can miss up to 2 modes)

In [51]:
Y_miss_2 = Y.clone().detach()

In [52]:
Y_miss_2[1000:1025, :params['p'][0]] = float('nan')
Y_miss_2[1020:1070, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
Y_miss_2[1065:1100, params['p'][0]+params['p'][1]:] = float('nan')
Y_miss_2[995:1010, params['p'][0]+params['p'][1]:] = float('nan')

In [53]:
Sigma_hat = Y_miss_2.nan_to_num().T @ Y_miss_2.nan_to_num() / Y_miss_2.shape[0]

In [54]:
W_new, L_new, Phi_new, _, _ = fit_EM_iter(Y_miss_2, Sigma_hat, W_init, L_init, Phi_init, maxit = 1000, impute_modes = True)

In [56]:
for W_m in W_new:
    print((W_m @ W_m.T)[:5, :5])

tensor([[10.3657,  1.0497, -3.5470, -3.3295, -1.6858],
        [ 1.0497,  6.8465,  1.1068,  2.1001, -4.6851],
        [-3.5470,  1.1068, 12.6371, -0.1547,  0.5544],
        [-3.3295,  2.1001, -0.1547, 16.6781, -4.6868],
        [-1.6858, -4.6851,  0.5544, -4.6868, 11.0375]])
tensor([[ 7.8186, -1.3460,  2.6567, -0.3209, -1.5927],
        [-1.3460, 10.1818,  0.7373, -2.2288,  1.4044],
        [ 2.6567,  0.7373,  9.3575, -1.1841, -0.5042],
        [-0.3209, -2.2288, -1.1841,  9.9197, -2.8929],
        [-1.5927,  1.4044, -0.5042, -2.8929,  4.5454]])
tensor([[ 8.0112,  3.9983, -2.9716, -0.1826, -3.5272],
        [ 3.9983,  6.0257,  1.2120, -1.1601, -1.0470],
        [-2.9716,  1.2120,  7.1393, -1.0845,  0.8766],
        [-0.1826, -1.1601, -1.0845,  5.4097, -0.4797],
        [-3.5272, -1.0470,  0.8766, -0.4797, 12.1412]])


In [57]:
for L_m in L_new:
    print((L_m @ L_m.T)[:5, :5])

tensor([[ 6.6449,  3.9203,  2.4455, -2.2563,  0.0761],
        [ 3.9203,  7.1838,  3.2836, -1.2351, -1.6313],
        [ 2.4455,  3.2836,  3.6937,  1.2251, -0.6588],
        [-2.2563, -1.2351,  1.2251,  3.4005,  0.0965],
        [ 0.0761, -1.6313, -0.6588,  0.0965,  2.2607]])
tensor([[ 3.7480,  3.9533, -2.4900,  1.8255, -2.6266],
        [ 3.9533,  6.8089, -3.4474,  2.5725, -0.1171],
        [-2.4900, -3.4474,  1.9097, -1.4142,  0.9209],
        [ 1.8255,  2.5725, -1.4142,  1.0479, -0.6311],
        [-2.6266, -0.1171,  0.9209, -0.6311,  4.5433]])
tensor([[ 2.0654, -2.1284,  0.6108, -1.1692, -0.0370],
        [-2.1284,  6.9454,  0.1160, -0.0162, -1.1002],
        [ 0.6108,  0.1160,  0.7263, -0.1975, -1.1027],
        [-1.1692, -0.0162, -0.1975,  1.3471, -0.4623],
        [-0.0370, -1.1002, -1.1027, -0.4623,  2.2449]])


In [58]:
for Phi_m in Phi_new:
    print(Phi_m[:5, :5])

tensor([[0.3227, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.2670, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3211, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.4105, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3483]])
tensor([[0.6751, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7978, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7157, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.7050, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7104]])
tensor([[0.5239, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4439, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5100, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.4919, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5146]])


Testing for missing data case (each sample missing max one mode)

In [122]:
metrics = {
    # 'WWt_rmse': [],
    'WWt_corr': [],
    # 'LLt_rmse': [],
    'LLt_corr': [],
    # 'Phi_rmse': [],
    'Phi_corr': [],
}

for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=True)
    Y = Y.T

    # insert missing data; missing modes do not overlap
    Y[1000:1025, :params['p'][0]] = float('nan')
    Y[1025:1075, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
    Y[1075:1100, params['p'][0]+params['p'][1]:] = float('nan')

    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=True)
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    LL = (torch.block_diag(*L_true)) @ (torch.block_diag(*L_true).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, L_new, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute_modes=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    LL_test = (torch.block_diag(*L_new)) @ (torch.block_diag(*L_new).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    # metrics['WWt_rmse'].append(torch.sqrt(torch.mean((WW_test - WW)**2)).item())
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    # metrics['LLt_rmse'].append(torch.sqrt(torch.mean((LL_test - LL)**2)).item())
    metrics['LLt_corr'].append(torch.corrcoef(
        torch.stack([LL.flatten(), LL_test.flatten()], dim=0)
    )[0,1].item())
    # metrics['Phi_rmse'].append(torch.sqrt(torch.mean((P_test - P)**2)).item())
    metrics['Phi_corr'].append(torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item())

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


1000 simulations took ~ 2m 26s to run

In [123]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9973
	Min: 0.9585
	Max: 0.9988
LLt_corr
	Mean: 0.999
	Min: 0.8493
	Max: 0.9998
Phi_corr
	Mean: 0.9507
	Min: 0.2004
	Max: 0.9865


Testing for missing data (samples may be missing up to 2/3 modes)

In [126]:
metrics = {
    # 'WWt_rmse': [],
    'WWt_corr': [],
    # 'LLt_rmse': [],
    'LLt_corr': [],
    # 'Phi_rmse': [],
    'Phi_corr': [],
}

for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=True)
    Y = Y.T

    # insert missing data; missing modes overlap
    Y[990:1025, :params['p'][0]] = float('nan')
    Y[1020:1070, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
    Y[1065:1100, params['p'][0]+params['p'][1]:] = float('nan')
    Y[980:1000, params['p'][0]+params['p'][1]:] = float('nan')

    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=True)
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    LL = (torch.block_diag(*L_true)) @ (torch.block_diag(*L_true).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, L_new, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute_modes=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    LL_test = (torch.block_diag(*L_new)) @ (torch.block_diag(*L_new).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    # metrics['WWt_rmse'].append(torch.sqrt(torch.mean((WW_test - WW)**2)).item())
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    # metrics['LLt_rmse'].append(torch.sqrt(torch.mean((LL_test - LL)**2)).item())
    metrics['LLt_corr'].append(torch.corrcoef(
        torch.stack([LL.flatten(), LL_test.flatten()], dim=0)
    )[0,1].item())
    # metrics['Phi_rmse'].append(torch.sqrt(torch.mean((P_test - P)**2)).item())
    metrics['Phi_corr'].append(torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item())

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


1000 simulations ran in ~ 2m 29.6s

In [127]:
for k, v in metrics.items():
    v = torch.tensor(v)
    print(k)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9974
	Min: 0.984
	Max: 0.9988
LLt_corr
	Mean: 0.9992
	Min: 0.9014
	Max: 0.9997
Phi_corr
	Mean: 0.9394
	Min: 0.2491
	Max: 0.9802
